# Neural Network Assignment: Customer Churn Prediction

## IBM Telco Customer Churn Dataset


- Part 1: Exploratory Data Analysis
- Part 2: Data Preprocessing
- Part 3: Build Baseline ANN
- Part 4: Hyperparameter Tuning
- Part 5: Regularization
- Part 6: Evaluation
- Part 7: Explainability
- Part 8: Error Analysis
- Part 9: Extra Validation
- Part 10: Reflection Questions



# Part 1: Exploratory Data Analysis

## 1.1 Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import zipfile
import os
import time
from pandas.plotting import scatter_matrix
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam, RMSprop, SGD
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

ModuleNotFoundError: No module named 'tensorflow'

## 1.2 Load Dataset

In [ ]:
csv_name = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
zip_name = "archive (2).zip"
if os.path.exists(csv_name):
    df = pd.read_csv(csv_name)
    print("Dataset loaded from CSV")
elif os.path.exists(zip_name):
    with zipfile.ZipFile(zip_name) as z:
        with z.open(csv_name) as f:
            df = pd.read_csv(f)
    print("Dataset loaded from ZIP")
else:
    raise FileNotFoundError("Keep archive (2).zip or WA_Fn-UseC_-Telco-Customer-Churn.csv in the same folder.")

## 1.3 Dataset Overview

In [ ]:
print("Dataset Shape:")
print(df.shape)
print("\nFirst 5 Rows:")
display(df.head())
print("\nColumn Names:")
print(df.columns.tolist())
print("\nData Types:")
print(df.dtypes)

## 1.4 Missing Values

In [ ]:
print("Missing Values:")
print(df.isnull().sum())
print("\nTotal Missing Values:")
print(df.isnull().sum().sum())
print("\nBlank Values in TotalCharges:")
print((df["TotalCharges"] == " ").sum())

## 1.5 Statistical Summary

In [ ]:
print("Numerical Summary:")
display(df.describe())
print("\nCategorical Summary:")
display(df.describe(include="object"))

## 1.6 Target Variable Analysis

In [ ]:
print("Churn Value Counts:")
print(df["Churn"].value_counts())
print("\nChurn Percentage:")
print(df["Churn"].value_counts(normalize=True) * 100)
plt.figure(figsize=(5,4))
df["Churn"].value_counts().plot(kind="bar")
plt.title("Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Count")
plt.show()
print("Interpretation:")
print("The target variable is Churn.")
print("Yes means the customer left the company.")
print("No means the customer stayed.")

## 1.7 Histograms

In [ ]:
df_temp = df.copy()
df_temp["TotalCharges"] = pd.to_numeric(df_temp["TotalCharges"], errors="coerce")
num_cols = df_temp.select_dtypes(include=["int64", "float64"]).columns.tolist()
for col in num_cols:
    plt.figure(figsize=(6,4))
    df_temp[col].hist(bins=30)
    plt.title("Histogram of " + col)
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.show()
print("Histograms show the distribution of numerical features.")

## 1.8 Boxplots

In [ ]:
for col in num_cols:
    plt.figure(figsize=(6,4))
    df_temp.boxplot(column=col)
    plt.title("Boxplot of " + col)
    plt.ylabel(col)
    plt.show()
print("Boxplots help identify outliers and spread of numerical features.")

## 1.9 Countplots

In [ ]:
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
cat_cols = [col for col in cat_cols if col not in ["customerID", "TotalCharges"]]
for col in cat_cols:
    plt.figure(figsize=(7,4))
    df[col].value_counts().plot(kind="bar")
    plt.title("Count Plot of " + col)
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.xticks(rotation=45)
    plt.show()
print("Countplots show category frequencies for categorical variables.")
print("customerID was skipped because it is only an identifier.")
print("TotalCharges was skipped here because it is converted to numeric for histograms and modeling.")

## 1.10 Correlation Heatmap

In [ ]:
corr_df = df_temp.copy()
corr_df["Churn"] = corr_df["Churn"].map({"No": 0, "Yes": 1})
corr = corr_df.select_dtypes(include=["int64", "float64"]).corr()
plt.figure(figsize=(8,6))
plt.imshow(corr)
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlation Heatmap")
plt.show()
print("The heatmap shows correlation among numerical features and churn.")

## 1.11 Pairplot of Selected Features

In [ ]:
selected_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
scatter_matrix(df_temp[selected_cols], figsize=(9,9), diagonal="hist")
plt.suptitle("Pairplot of Selected Numerical Features")
plt.show()
print("Pairplots show relationships between selected numerical features.")

## 1.12 EDA Questions

In [ ]:
print("EDA Questions and Answers:")
print("1. What is the target variable? Churn.")
print("2. What does churn mean? It means a customer left the company.")
print("3. Are there missing values? Regular missing values are checked using isnull().")
print("4. Why check TotalCharges separately? It has blank spaces that must be converted to missing values.")
print("5. Is churn balanced? No, there are more non-churn customers than churn customers.")
print("6. Why use visualizations? They help understand distributions, outliers, and relationships.")

# Part 2: Data Preprocessing

## 2.1 Copy Data and Convert TotalCharges

In [ ]:
df_clean = df.copy()
df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce")
print("TotalCharges converted to numeric")
print("Missing TotalCharges after conversion:")
print(df_clean["TotalCharges"].isnull().sum())

## 2.2 Handle Missing Values

In [ ]:
for col in df_clean.columns:
    if df_clean[col].dtype == "object":
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])
    else:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())
print("Missing Values After Treatment:")
print(df_clean.isnull().sum().sum())

## 2.3 Feature Engineering

In [ ]:
df_clean["AvgChargesPerMonth"] = df_clean["TotalCharges"] / (df_clean["tenure"] + 1)
df_clean["HasInternetService"] = np.where(df_clean["InternetService"] == "No", 0, 1)
df_clean["IsLongTermCustomer"] = np.where(df_clean["tenure"] >= 24, 1, 0)
service_cols = ["PhoneService", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
df_clean["NumServices"] = 0
for col in service_cols:
    df_clean["NumServices"] = df_clean["NumServices"] + np.where(df_clean[col].isin(["Yes"]), 1, 0)
print("Feature Engineering Completed")
print("New Features:")
print(["AvgChargesPerMonth", "HasInternetService", "IsLongTermCustomer", "NumServices"])

## 2.4 Encode Target Variable

In [ ]:
df_clean["Churn"] = df_clean["Churn"].map({"No": 0, "Yes": 1})
print("Target encoded")
print(df_clean["Churn"].value_counts())

## 2.5 Drop Customer ID

In [ ]:
df_clean = df_clean.drop("customerID", axis=1)
print("customerID removed because it is only an identifier and not useful for prediction")

## 2.6 Encode Categorical Variables

In [ ]:
X = df_clean.drop("Churn", axis=1)
y = df_clean["Churn"]
X = pd.get_dummies(X, drop_first=True)
print("Categorical Encoding Completed")
print("Feature Shape:")
print(X.shape)

## 2.7 Train-Test Split 80/20

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print("Training Shape:")
print(X_train.shape)
print("\nTesting Shape:")
print(X_test.shape)
print("\nTraining Target Distribution:")
print(y_train.value_counts(normalize=True) * 100)
print("\nTesting Target Distribution:")
print(y_test.value_counts(normalize=True) * 100)

## 2.8 Scale Numeric Features

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X.columns)
print("Scaling Completed")
display(X_train_scaled_df.head())

## 2.9 Class Weights

In [ ]:
classes = np.unique(y_train)
class_weights_array = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights = dict(zip(classes, class_weights_array))
print("Class Weights:")
print(class_weights)
print("Class weights help the ANN pay more attention to the minority churn class.")

# Part 3: Build Baseline ANN

## 3.1 Define Baseline ANN Model

In [ ]:
tf.random.set_seed(42)
baseline_model = Sequential()
baseline_model.add(Dense(32, activation="relu", input_shape=(X_train_scaled.shape[1],)))
baseline_model.add(Dense(16, activation="relu"))
baseline_model.add(Dense(1, activation="sigmoid"))
baseline_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
baseline_model.summary()

## 3.2 Train Baseline ANN

In [ ]:
baseline_history = baseline_model.fit(X_train_scaled, y_train, validation_split=0.20, epochs=50, batch_size=32, verbose=0)
print("Baseline ANN training completed")

## 3.3 Plot Baseline Accuracy

In [ ]:
plt.figure(figsize=(7,5))
plt.plot(baseline_history.history["accuracy"], label="Training Accuracy")
plt.plot(baseline_history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Baseline Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

## 3.4 Plot Baseline Loss

In [ ]:
plt.figure(figsize=(7,5))
plt.plot(baseline_history.history["loss"], label="Training Loss")
plt.plot(baseline_history.history["val_loss"], label="Validation Loss")
plt.title("Baseline Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

## 3.5 Baseline Model Evaluation

In [ ]:
baseline_prob = baseline_model.predict(X_test_scaled).ravel()
baseline_pred = (baseline_prob >= 0.5).astype(int)
print("Baseline Accuracy:", accuracy_score(y_test, baseline_pred))
print("Baseline Precision:", precision_score(y_test, baseline_pred))
print("Baseline Recall:", recall_score(y_test, baseline_pred))
print("Baseline F1:", f1_score(y_test, baseline_pred))
print("Baseline ROC-AUC:", roc_auc_score(y_test, baseline_prob))
print("Baseline Confusion Matrix:")
print(confusion_matrix(y_test, baseline_pred))

# Part 4: Hyperparameter Tuning

## 4.1 Create Model Function

In [ ]:
def create_ann(hidden_layers=[32, 16], activation="relu", optimizer_name="adam", learning_rate=0.001, dropout_rate=0.0, l2_rate=0.0, batch_norm=False):
    model = Sequential()
    regularizer = l2(l2_rate) if l2_rate > 0 else None
    for i, neurons in enumerate(hidden_layers):
        if i == 0:
            model.add(Dense(neurons, activation=activation, kernel_regularizer=regularizer, input_shape=(X_train_scaled.shape[1],)))
        else:
            model.add(Dense(neurons, activation=activation, kernel_regularizer=regularizer))
        if batch_norm:
            model.add(BatchNormalization())
        if dropout_rate > 0:
            model.add(Dropout(dropout_rate))
    model.add(Dense(1, activation="sigmoid"))
    if optimizer_name == "adam":
        optimizer = Adam(learning_rate=learning_rate)
    elif optimizer_name == "rmsprop":
        optimizer = RMSprop(learning_rate=learning_rate)
    else:
        optimizer = SGD(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"])
    return model
print("Model creation function ready")

## 4.2 Compare Hyperparameters

In [ ]:
tuning_configs = [
    {"name": "Model 1", "hidden_layers": [32, 16], "activation": "relu", "optimizer_name": "adam", "learning_rate": 0.001},
    {"name": "Model 2", "hidden_layers": [64, 32], "activation": "relu", "optimizer_name": "adam", "learning_rate": 0.001},
    {"name": "Model 3", "hidden_layers": [32, 16], "activation": "tanh", "optimizer_name": "adam", "learning_rate": 0.001},
    {"name": "Model 4", "hidden_layers": [64, 32], "activation": "relu", "optimizer_name": "rmsprop", "learning_rate": 0.001},
    {"name": "Model 5", "hidden_layers": [32, 16], "activation": "relu", "optimizer_name": "adam", "learning_rate": 0.0005}
]
tuning_results = []
best_tuning_model = None
best_tuning_f1 = 0
for config in tuning_configs:
    model = create_ann(hidden_layers=config["hidden_layers"], activation=config["activation"], optimizer_name=config["optimizer_name"], learning_rate=config["learning_rate"])
    start_time = time.time()
    model.fit(X_train_scaled, y_train, validation_split=0.20, epochs=15, batch_size=32, verbose=0)
    training_time = time.time() - start_time
    prob = model.predict(X_test_scaled, verbose=0).ravel()
    pred = (prob >= 0.5).astype(int)
    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred)
    recall = recall_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    roc_auc = roc_auc_score(y_test, prob)
    tuning_results.append([config["name"], str(config["hidden_layers"]), config["activation"], config["optimizer_name"], config["learning_rate"], accuracy, precision, recall, f1, roc_auc, training_time])
    if f1 > best_tuning_f1:
        best_tuning_f1 = f1
        best_tuning_model = model
tuning_table = pd.DataFrame(tuning_results, columns=["Model", "Hidden Layers", "Activation", "Optimizer", "Learning Rate", "Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "Training Time"])
display(tuning_table)

## 4.3 Hyperparameter Tuning Interpretation

In [ ]:
print("Hyperparameter Tuning Interpretation:")
print("Different hidden layers, neurons, activations, optimizers, and learning rates were compared.")
print("The best model can be selected using F1-score because churn prediction needs balance between precision and recall.")
print("ROC-AUC is also useful because it measures how well the model separates churn and non-churn customers.")

# Part 5: Regularization

## 5.1 Build Regularized ANN

In [ ]:
regularized_model = create_ann(hidden_layers=[64, 32], activation="relu", optimizer_name="adam", learning_rate=0.001, dropout_rate=0.30, l2_rate=0.001, batch_norm=True)
regularized_model.summary()

## 5.2 Train with Dropout, EarlyStopping, L2, Batch Normalization, LR Scheduler, and Class Weights

In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
lr_scheduler = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4)
regularized_history = regularized_model.fit(X_train_scaled, y_train, validation_split=0.20, epochs=50, batch_size=32, callbacks=[early_stop, lr_scheduler], class_weight=class_weights, verbose=0)
print("Regularized ANN training completed")

## 5.3 Plot Regularized Accuracy

In [ ]:
plt.figure(figsize=(7,5))
plt.plot(regularized_history.history["accuracy"], label="Training Accuracy")
plt.plot(regularized_history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Regularized Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

## 5.4 Plot Regularized Loss

In [ ]:
plt.figure(figsize=(7,5))
plt.plot(regularized_history.history["loss"], label="Training Loss")
plt.plot(regularized_history.history["val_loss"], label="Validation Loss")
plt.title("Regularized Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

## 5.5 Regularization Explanation

In [ ]:
print("Regularization Explanation:")
print("Dropout randomly turns off some neurons during training to reduce overfitting.")
print("EarlyStopping stops training when validation loss stops improving.")
print("L2 regularization penalizes large weights.")
print("Batch Normalization stabilizes training.")
print("Learning rate scheduler reduces learning rate when validation loss stops improving.")
print("Class weights help the model learn the minority churn class.")

# Part 6: Evaluation

## 6.1 Evaluate Final ANN

In [ ]:
final_model = regularized_model
final_prob = final_model.predict(X_test_scaled).ravel()
final_pred = (final_prob >= 0.5).astype(int)
final_accuracy = accuracy_score(y_test, final_pred)
final_precision = precision_score(y_test, final_pred)
final_recall = recall_score(y_test, final_pred)
final_f1 = f1_score(y_test, final_pred)
final_roc_auc = roc_auc_score(y_test, final_prob)
print("Final Accuracy:", final_accuracy)
print("Final Precision:", final_precision)
print("Final Recall:", final_recall)
print("Final F1:", final_f1)
print("Final ROC-AUC:", final_roc_auc)

## 6.2 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, final_pred)
print("Confusion Matrix:")
print(cm)
tn, fp, fn, tp = cm.ravel()
print("True Negatives:", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives:", tp)

## 6.3 Classification Report

In [ ]:
print("Classification Report:")
print(classification_report(y_test, final_pred))

## 6.4 Baseline vs Final Comparison

In [ ]:
comparison_table = pd.DataFrame({
    "Model": ["Baseline ANN", "Regularized ANN"],
    "Accuracy": [accuracy_score(y_test, baseline_pred), final_accuracy],
    "Precision": [precision_score(y_test, baseline_pred), final_precision],
    "Recall": [recall_score(y_test, baseline_pred), final_recall],
    "F1": [f1_score(y_test, baseline_pred), final_f1],
    "ROC-AUC": [roc_auc_score(y_test, baseline_prob), final_roc_auc]
})
display(comparison_table)

# Part 7: Explainability

## 7.1 Permutation Importance

In [ ]:
baseline_auc = roc_auc_score(y_test, final_prob)
rng = np.random.default_rng(42)
importance_results = []
for col in X_test_scaled_df.columns:
    X_perm = X_test_scaled_df.copy()
    X_perm[col] = rng.permutation(X_perm[col].values)
    perm_prob = final_model.predict(X_perm.values, verbose=0).ravel()
    perm_auc = roc_auc_score(y_test, perm_prob)
    importance = baseline_auc - perm_auc
    importance_results.append([col, importance])
importance_df = pd.DataFrame(importance_results, columns=["Feature", "Permutation Importance"])
importance_df = importance_df.sort_values(by="Permutation Importance", ascending=False)
display(importance_df.head(15))

## 7.2 Plot Top Feature Importances

In [ ]:
top_importance = importance_df.head(10).sort_values(by="Permutation Importance")
plt.figure(figsize=(8,6))
plt.barh(top_importance["Feature"], top_importance["Permutation Importance"])
plt.title("Top 10 Permutation Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

## 7.3 Explainability Interpretation

In [ ]:
print("Explainability Interpretation:")
print("Permutation importance checks how much model performance drops when a feature is shuffled.")
print("If shuffling a feature reduces ROC-AUC strongly, that feature is important.")
print("This helps understand which customer attributes influence churn prediction.")

# Part 8: Error Analysis

## 8.1 Create Error DataFrame

In [ ]:
error_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": final_pred,
    "Churn_Probability": final_prob
})
error_df["Error_Type"] = "Correct"
error_df.loc[(error_df["Actual"] == 0) & (error_df["Predicted"] == 1), "Error_Type"] = "False Positive"
error_df.loc[(error_df["Actual"] == 1) & (error_df["Predicted"] == 0), "Error_Type"] = "False Negative"
display(error_df.head())

## 8.2 Analyze False Positives and False Negatives

In [ ]:
print("Error Type Counts:")
print(error_df["Error_Type"].value_counts())
false_positives = error_df[error_df["Error_Type"] == "False Positive"]
false_negatives = error_df[error_df["Error_Type"] == "False Negative"]
print("\nFalse Positives:")
display(false_positives.head())
print("\nFalse Negatives:")
display(false_negatives.head())

## 8.3 Error Analysis Interpretation

In [ ]:
print("Error Analysis Interpretation:")
print("False Positive means the model predicted churn but the customer did not churn.")
print("False Negative means the model predicted no churn but the customer actually churned.")
print("False negatives are more serious because the company misses customers who are likely to leave.")
print("Reducing false negatives can help improve customer retention.")

# Part 9: Extra Validation

## 9.1 K-Fold Validation

In [ ]:
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
fold_results = []
for fold, (train_index, valid_index) in enumerate(skf.split(X_train_scaled, y_train), 1):
    X_fold_train = X_train_scaled[train_index]
    X_fold_valid = X_train_scaled[valid_index]
    y_fold_train = y_train.iloc[train_index]
    y_fold_valid = y_train.iloc[valid_index]
    fold_model = create_ann(hidden_layers=[32, 16], activation="relu", optimizer_name="adam", learning_rate=0.001)
    fold_model.fit(X_fold_train, y_fold_train, epochs=10, batch_size=32, verbose=0)
    fold_prob = fold_model.predict(X_fold_valid, verbose=0).ravel()
    fold_pred = (fold_prob >= 0.5).astype(int)
    fold_f1 = f1_score(y_fold_valid, fold_pred)
    fold_results.append(fold_f1)
print("K-Fold F1 Scores:")
print(fold_results)
print("Average K-Fold F1 Score:")
print(np.mean(fold_results))

## 9.2 Save Trained Model

In [ ]:
final_model.save("customer_churn_ann_model.keras")
print("Trained model saved as customer_churn_ann_model.keras")

## 9.3 Extra Validation Summary

In [ ]:
print("Extra Validation Summary:")
print("Batch Normalization was used in the regularized model.")
print("Learning rate scheduler was used to reduce the learning rate when validation loss stopped improving.")
print("Class weights were used to handle churn imbalance.")
print("Feature engineering was used to create customer behavior features.")
print("K-fold validation was used to check model stability.")

# Part 10: Reflection Questions

In [ ]:
print("Reflection Questions and Answers:")
print("1. Why is scaling required for ANN?")
print("Answer: Scaling helps neural networks train faster and more stable.")
print("\n2. What is ReLU?")
print("Answer: ReLU is an activation function that outputs 0 for negative values and keeps positive values.")
print("\n3. Why is sigmoid used in the output layer?")
print("Answer: Sigmoid gives probability between 0 and 1 for binary classification.")
print("\n4. ANN vs Logistic Regression?")
print("Answer: Logistic Regression is linear, while ANN can learn non-linear relationships.")
print("\n5. What is overfitting?")
print("Answer: Overfitting happens when the model learns training data too well but performs poorly on new data.")
print("\n6. What does Dropout do?")
print("Answer: Dropout randomly turns off neurons during training to reduce overfitting.")
print("\n7. What is Adam optimizer?")
print("Answer: Adam is an optimizer that adjusts learning rates during training.")
print("\n8. What is learning rate?")
print("Answer: Learning rate controls how big the model weight updates are.")
print("\n9. What is ROC-AUC?")
print("Answer: ROC-AUC measures how well the model separates churn and non-churn customers.")
print("\n10. What are deployment considerations?")
print("Answer: The model should be monitored, updated with new data, and checked for fairness and performance.")

# Final Conclusion


The workflow included:

- EDA
- Missing value handling
- Feature engineering
- Encoding
- Scaling
- 80/20 train-test split
- Baseline ANN
- Hyperparameter tuning
- Dropout, EarlyStopping, L2 regularization
- Batch Normalization and learning rate scheduler
- Classification evaluation
- Permutation importance
- Error analysis
- K-fold validation
- Model saving
- Reflection questions

The ANN model helps predict customers likely to churn so the telecom company can take retention actions.
